In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
import subprocess
import time

# Step 1 - Install zstd first
subprocess.run("apt-get install -y zstd", shell=True, check=True)

# Step 2 - install ollama
subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    check=True
)

# Step 3 - Start ollama server in background
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(15)  # Wait for server to start

# Step 4 - Pull the model
subprocess.run(["ollama", "pull", "llama3.2:3b"], check=True)

print("Ollama is ready!")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (11.2 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 6.9 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   2% ▕                  ▏  43 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   7% ▕█                 ▏ 134 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  12% ▕██                ▏ 233 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  14% ▕██                ▏ 292 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  19% ▕███               ▏ 382 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  24% ▕████              ▏ 482 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  26% ▕████              ▏ 532 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  31% ▕█████             ▏ 631 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  36% ▕█████

Ollama is ready!


pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [3]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [4]:
import ollama

response = ollama.chat(
    model='llama3.2:3b',
    messages=[{'role': 'user', 'content': 'Say hello in one sentence'}]
)

print(response['message']['content'])

Hello!


In [5]:
# Load dataset (2WikiMultihopQA dev)
import requests, zipfile, io, json

url = "https://gitlab.tudelft.nl/anonymous_arr/bcqa_data/-/raw/main/2wikimultihopQA.zip"
response = requests.get(url)

z = zipfile.ZipFile(io.BytesIO(response.content))

with z.open("2wikimultihopQA/dev.json") as f:
    data = json.load(f)

# take 500 queries
data = data[:500]

print("Loaded queries:", len(data))

Loaded queries: 500


In [6]:
pip install gdown

Note: you may need to restart the kernel to use updated packages.


In [7]:
import gdown

url = "https://drive.google.com/uc?id=1aIyo583DCy3oAuQSgvIKf_SUnV3PEAer"
output = "bm25_2wiki.json"

gdown.download(url, output, quiet=False)

# load
with open(output) as f:
    bm25_data = json.load(f)

print(type(bm25_data))
print(len(bm25_data))

# check structure
first_key = list(bm25_data.keys())[0]
print("Example key:", first_key)
print("Example docs:", bm25_data[first_key][:2])

Downloading...
From: https://drive.google.com/uc?id=1aIyo583DCy3oAuQSgvIKf_SUnV3PEAer
To: /kaggle/working/bm25_2wiki.json
100%|██████████| 52.7M/52.7M [00:00<00:00, 134MB/s] 


<class 'dict'>
1200
Example key: 8813f87c0bdd11eba7f7acde48001122
Example docs: ['Polish-Russian War (Wojna polsko-ruska) is a 2009 Polish film directed by Xawery Żuławski based on the novel Polish-Russian War under the white-red flag by Dorota Masłowska.', 'Polish- Russian War( Wojna polsko- ruska) is a 2009 Polish film directed by Xawery Żuławski based on the novel Polish- Russian War under the white- red flag by Dorota Masłowska.']


In [8]:
# Check if dataset IDs match BM25 keys

sample_item = data[0]

print("Dataset ID:", sample_item.get("_id", sample_item.get("id")))
print("Exists in BM25:", sample_item.get("_id", sample_item.get("id")) in bm25_data)

# Check a few samples
for i in range(5):
    qid = data[i].get("_id", data[i].get("id"))
    print(qid, "->", qid in bm25_data)

Dataset ID: 8813f87c0bdd11eba7f7acde48001122
Exists in BM25: True
8813f87c0bdd11eba7f7acde48001122 -> True
61a46987092f11ebbdaeac1f6bf848b6 -> True
e2a3bf2a0bdd11eba7f7acde48001122 -> True
0cd3bdea0bde11eba7f7acde48001122 -> True
f9dcb4a60bda11eba7f7acde48001122 -> True


In [9]:
pip install -U transformers sentence-transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 79.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.5 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: tran

In [10]:
from sentence_transformers import CrossEncoder

cross_enc = CrossEncoder("nreimers/mmarco-mMiniLMv2-L12-H384-v1",device="cuda",trust_remote_code=True)

config.json:   0%|          | 0.00/813 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [11]:
def rerank_docs(query, docs, top_k=5):
    pairs = [(query, doc) for doc in docs]
    scores = cross_enc.predict(pairs)
    ranked = sorted(zip(scores, docs), reverse=True)
    return [doc for _, doc in ranked[:top_k]]

print("CrossEncoder ready (GPU)")

CrossEncoder ready (GPU)


In [18]:
# Not needed
def recall_at_k_2wiki(sample, retrieved_docs, k=5):
    supporting_facts = [fact[0] for fact in sample["supporting_facts"]]
    gold_passages = [item for item in sample['context'] if item[0] in supporting_facts]
    gold_items = list(set([" ".join(item[1]) for item in gold_passages]))

    retrieved_items = list(set(retrieved_docs[:k]))

    count = 0
    for doc in retrieved_items:
        if doc in gold_items:
            count += 1

    return count / len(gold_items)

print("Recall@k (2Wiki) ready")

Recall@k (2Wiki) ready


In [ ]:
import re
# Baseline follow-up generation (plain prompting)

def ensure_question(text, query):
    text = str(text).strip()
    text = text.replace("Question:", "").replace("Follow-up", "").replace(":", "").strip()
    text = " ".join(text.split())

    if not text:
        return f"What specific information are you looking for about {query}?"

    if "?" in text:
        text = text.split("?")[0].strip() + "?"
        return text

    lowered = text.lower()
    if lowered.startswith(("what ", "which ", "who ", "where ", "when ", "why ", "how ", "are ", "is ", "do ", "does ", "did ", "can ", "could ", "would ", "should ")):
        return text.rstrip(".!") + "?"

    if lowered.endswith(" is") or lowered.endswith(" are"):
        return text.rstrip(".!") + "?"

    return f"What specific information are you looking for about {query}?"


def generate_baseline_followup(query):
    prompt = f"""
    Generate ONE useful follow-up question to clarify the query.

    Rules:
    - Ask about a specific missing detail (e.g., location, type, features, time, etc.)
    - Do NOT ask vague questions like "What do you want to know?"
    - The question must narrow down the query and be relevant to the original query and potential information needs.
    - Frame the output as a question.
    - The output must end with a question mark.
    - Do not answer the query.
    - Do not complete the sentence as a statement.

    Query: {query}

    Return ONLY the question.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    msg = response['message']
    text = msg.get('content', '').strip()

    return ensure_question(text, query)


In [ ]:
# Baseline: Decomposition (Parallel)
def generate_decomposition_followup(query):
    prompt = f"""
    Generate 3 different follow-up questions about the query, each focusing on a different aspect.

    Examples of aspects:
    - what/who it is
    - role, function, or purpose
    - history, location, or usage

    Rules:
    - Each question must be specific and useful
    - Do NOT ask generic questions like "What do you want to know?"
    - Do NOT repeat the same idea
    - Keep questions simple and relevant

    Query: {query}

    Return ONLY 3 questions, each on a new line.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 100, 'temperature': 0}
    )

    text = response['message']['content'].strip()

    # split lines and pick first valid question
    lines = [re.sub(r"^\d+[\.\)]\s*", "", l.strip()) for l in text.split("\n") if l.strip()]

    valid = []
    for l in lines:
        q = ensure_question(l, query)

        # remove generic fallback
        if "what specific information" in q.lower():
            continue

        # avoid super short junk
        if len(q.split()) < 4:
            continue

        valid.append(q)

    if valid:
        return valid[0]

    return ensure_question("", query)

In [ ]:
# Baseline: Yes/No (Claim-style)
def generate_yesno_followup(query):
    prompt = f"""
    Generate ONE yes/no follow-up question to clarify the query.

    Rules:
    - The question must be answerable with ONLY "yes" or "no"
    - Do NOT ask multiple-choice questions
    - Do NOT include "or" comparisons (e.g., person, place, or thing)
    - Do NOT assume specific facts
    - Keep it simple and relevant

    Query: {query}

    Return ONLY the question.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    text = response['message']['content'].strip()

    if " or " in text.lower():
        return ensure_question("", query)

    return ensure_question(text, query)

In [ ]:
test_sample = data[:5]

for item in test_sample:
    qid = item.get("_id", item.get("id"))
    query = item["question"]

    print("\nQUERY:", query)

    # Baseline
    b = generate_baseline_followup(query)
    print("Baseline:", b)

    # Decomposition
    d = generate_decomposition_followup(query)
    print("Decomposition:", d)

    # Yes/No
    y = generate_yesno_followup(query)
    print("Yes/No:", y)


QUERY: Who is the mother of the director of film Polish-Russian War (Film)?
Baseline: Is the director of the film "Polish-Russian War" (Film) a well-known or prominent filmmaker?
Decomposition: What is the name of the director of the film "Polish-Russian War"?
Yes/No: Is the director of the film Polish-Russian War a female?

QUERY: Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
Baseline: Was the release year of both films specified in the original query?
Decomposition: What is the release year of Blind Shaft?
Yes/No: Was The Mask of Fu Manchu released before 1931?

QUERY: When did John V, Prince Of Anhalt-Zerbst's father die?
Baseline: Was John V, Prince Of Anhalt-Zerbst's father specifically Frederick Louis, Duke of Saxe-Gotha-Altenburg?
Decomposition: What was the name of John V, Prince Of Anhalt-Zerbst's wife?
Yes/No: Was John V, Prince Of Anhalt-Zerbst a historical figure?

QUERY: What is the award that the director of film Wearing Velvet Slippers Under A Golden 

In [ ]:
test_sample = data[:5]

for item in test_sample:
    qid = item.get("_id", item.get("id"))
    query = item["question"]

    raw_docs = bm25_data[qid]

    print("\nQUERY:", query)

    # BM25
    bm25_score = recall_at_k_2wiki(item, raw_docs)
    print("BM25 Recall@5:", bm25_score)

    # Baseline
    bq = generate_baseline_followup(query)
    b_docs = rerank_docs(bq, raw_docs)
    print("Baseline Recall@5:", recall_at_k_2wiki(item, b_docs))

    # Decomposition
    dq = generate_decomposition_followup(query)
    d_docs = rerank_docs(dq, raw_docs)
    print("Decomposition Recall@5:", recall_at_k_2wiki(item, d_docs))

    # Yes/No
    yq = generate_yesno_followup(query)
    y_docs = rerank_docs(yq, raw_docs)
    print("Yes/No Recall@5:", recall_at_k_2wiki(item, y_docs))


QUERY: Who is the mother of the director of film Polish-Russian War (Film)?
BM25 Recall@5: 0.5
Baseline Recall@5: 0.5
Decomposition Recall@5: 0.5
Yes/No Recall@5: 0.5

QUERY: Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
BM25 Recall@5: 0.5
Baseline Recall@5: 0.0
Decomposition Recall@5: 0.5
Yes/No Recall@5: 0.5

QUERY: When did John V, Prince Of Anhalt-Zerbst's father die?
BM25 Recall@5: 0.5
Baseline Recall@5: 0.0
Decomposition Recall@5: 0.5
Yes/No Recall@5: 0.5

QUERY: What is the award that the director of film Wearing Velvet Slippers Under A Golden Umbrella won?
BM25 Recall@5: 0.5
Baseline Recall@5: 0.5
Decomposition Recall@5: 0.5
Yes/No Recall@5: 0.5

QUERY: Where was the director of film Ronnie Rocket born?
BM25 Recall@5: 0.5
Baseline Recall@5: 0.0
Decomposition Recall@5: 0.0
Yes/No Recall@5: 0.5


In [12]:
def extract_aspects(query):
    import re

    query_clean = str(query).strip()
    query_lower = query_clean.lower()
    words = [w for w in re.findall(r"[a-zA-Z0-9]+", query_clean) if len(w) > 1]

    def has_any(phrases):
        return any(p in query_lower for p in phrases)

    def classify_query_type(text):
        text_lower = text.lower().strip()
        tokens = [w for w in re.findall(r"[a-zA-Z0-9]+", text_lower) if len(w) > 1]

        if len(tokens) <= 2:
            return "ambiguous_short_query"

        if has_any(["family tree", "ancestry", "genealogy"]):
            return "family_relationship"

        if has_any(["hotel", "resort", "inn", "airport", "terminal", "airlines"]):
            return "place_or_facility"

        if has_any(["restaurant", "museum", "university", "company", "organization", "school", "farm"]):
            return "organization_or_place"

        if has_any(["cheap", "best", "budget", "affordable", "price", "pricing", "cost", "provider", "plan"]):
            return "service_or_product"

        if has_any(["map", "located", "location", "address", "directions", "where is"]):
            return "place_or_location"

        if has_any(["who won", "biography", "born", "died", "who is", "who was"]):
            return "person"

        # Broad person-name heuristic without memorizing topic-specific exceptions.
        title_cased = [part for part in re.split(r"\s+", text.strip()) if part]
        if 2 <= len(title_cased) <= 4 and all(part[:1].isalpha() for part in title_cased):
            non_person_markers = {
                "airport", "hotel", "resort", "museum", "restaurant", "company",
                "university", "school", "farm", "lake", "park", "internet",
                "family", "tree", "map", "disease", "county", "tourism"
            }
            if not any(token in non_person_markers for token in tokens):
                return "person"

        if has_any(["disease", "symptoms", "treatment", "causes"]):
            return "topic_or_condition"

        if has_any(["album", "movie", "film", "book", "song", "game"]):
            return "work_or_media"

        return "generic"

    query_type = classify_query_type(query_clean)

    aspect_priors = {
        "family_relationship": ["family members", "ancestry", "relationships"],
        "service_or_product": ["type or option", "pricing or availability", "features or providers"],
        "place_or_facility": ["location", "services or amenities", "current status"],
        "place_or_location": ["location", "key features", "relevance or use"],
        "organization_or_place": ["main purpose or type", "location", "notable features"],
        "person": ["identity", "role or career", "notable contribution"],
        "topic_or_condition": ["definition or type", "causes or features", "diagnosis or treatment"],
        "work_or_media": ["work identity", "content or subject", "notable significance"],
        "ambiguous_short_query": ["intended meaning", "context", "specific need"],
    }

    if query_type in aspect_priors:
        return aspect_priors[query_type]

    prompt = f"""
    Identify 3 key aspects needed to answer this query well.

    An aspect is a dimension of information required to satisfy the query,
    including implicit missing information when relevant.

    Rules:
    - exactly 3 aspects
    - short phrases only
    - 2 to 5 words each
    - no full sentences
    - avoid generic items like "details", "information", "major achievements", or "historical significance"
    - prefer aspects that help generate useful follow-up questions for this specific query type
    - if the query is about a person, use person-style aspects only when appropriate
    - if the query is about a place, organization, service, or object, do NOT use person-style aspects

    Query: {query_clean}

    Aspects:
    """
    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 80, 'temperature': 0}
    )

    def clean_aspects(text):
        lines = text.split("\n")
        cleaned = []
        banned = {
            "details", "information", "general information", "overview", "background",
            "major achievements", "historical significance"
        }

        for line in lines:
            line = line.strip()
            line = line.lstrip("1234567890.-) ").strip()
            line = re.sub(r'[^a-zA-Z0-9\s/&-]', '', line).strip()

            if not line:
                continue
            if len(line.split()) < 1 or len(line.split()) > 6:
                continue
            if line.lower() in banned:
                continue
            if any(x in line.lower() for x in ["part", "definition", "usage"]):
                continue

            cleaned.append(line)

        deduped = []
        seen = set()
        for item in cleaned:
            key = item.lower()
            if key not in seen:
                seen.add(key)
                deduped.append(item)

        return deduped[:3]

    raw = response['message']['content']
    aspects = clean_aspects(raw)

    if len(aspects) < 3:
        fallback = ["main topic", "key facts", "context"]
        for item in fallback:
            if len(aspects) < 3 and item.lower() not in {a.lower() for a in aspects}:
                aspects.append(item)

    return aspects[:3]

In [13]:
# detect ambiguity BEFORE answer generation
def detect_ambiguity(query):
    query = query.strip()

    # well-formed named entities usually do not need intent clarification
    if len(query.split()) >= 3 and query.replace(" ", "").isalpha():
        return False

    prompt = f"""
    Query: {query}

    Does this query have multiple meanings or interpretations,
    or is it missing the intent needed to answer it correctly?

    Answer only: YES or NO
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 5, 'temperature': 0}
    )

    return "YES" in response['message']['content'].upper()

In [14]:
# Missing aspect identification
def find_missing_aspect(query, answer, aspects, allow_intent=False):

    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    answer_norm = normalize(answer)

    if allow_intent and detect_ambiguity(query):
        return "INTENT"

    weak_answer = (
        not answer_norm
        or "insufficient information" in answer_norm
        or "ambiguous query" in answer_norm
        or len(answer_norm.split()) < 12
        or not str(answer).strip().endswith((".", "?", "!"))
    )

    if not aspects:
        return "DETAIL" if weak_answer else "NONE"

    stopwords = {"and", "of", "the", "to", "from", "for", "in", "on", "with", "or"}
    coverage = {}
    uncovered = []

    for aspect in aspects:
        aspect_norm = normalize(aspect)
        aspect_words = [w for w in aspect_norm.split() if w not in stopwords and len(w) > 2]

        if not aspect_words:
            coverage[aspect] = 0.0
            uncovered.append(aspect)
            continue

        matched = sum(1 for w in aspect_words if w in answer_norm)
        ratio = matched / max(len(aspect_words), 1)
        coverage[aspect] = ratio

        if ratio < 0.6:
            uncovered.append(aspect)

    if not uncovered:
        return "DETAIL" if weak_answer else "NONE"

    if weak_answer and len(uncovered) == len(aspects):
        return uncovered[0]

    prompt = f"""
    Original query: {query}

    Combined answer so far: {answer}

    Candidate aspects and approximate coverage:
    {coverage}

    Remaining candidate aspects:
    {uncovered}

    Choose the ONE most important aspect still missing to answer the original query well.

    Rules:
    - Return NONE only if the answer is already sufficient.
    - Otherwise return exactly one aspect from the remaining candidate list.
    - Prefer the aspect whose absence most limits answering the original query.
    - Do not invent a new aspect.

    Output format:
    Missing: <NONE or exact aspect>
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 30, 'temperature': 0}
    )

    text = response['message']['content'].strip()
    if "Missing:" in text:
        missing = text.split("Missing:")[1].split("\n")[0].strip()
    else:
        missing = uncovered[0]

    if missing == "NONE":
        return "DETAIL" if weak_answer else "NONE"

    if missing in uncovered:
        return missing

    return uncovered[0]

In [15]:
# Improved follow-up generation
def generate_aspect_followup(query, missing_aspect, aspects=None, asked_followups=None):
    asked_followups = asked_followups or []
    asked_norm = {q.strip().lower() for q in asked_followups if q}

    if missing_aspect == "NONE":
        return None

    if missing_aspect == "INTENT":
        candidate = f"What specific information do you want to know about {query}?"
        return None if candidate.lower() in asked_norm else candidate

    if missing_aspect == "DETAIL":
        candidate = f"What specific detail about {query} are you looking for?"
        return None if candidate.lower() in asked_norm else candidate

    prompt = f"""
    Original query: {query}

    Missing aspect: {missing_aspect}

    Generate ONE clear, direct clarification question asking only about this missing aspect.

    Rules:
    - Start with What, Which, Who, Where, or When.
    - Keep it specific and factual.
    - Mention the original topic naturally.
    - Do not ask a generic question.
    - Do not repeat the aspect wording awkwardly.
    - Output only the question.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    text = response['message']['content'].strip()
    if "?" in text:
        text = text.split("?")[0].strip() + "?"
    elif text:
        text = text.strip() + "?"

    if not text:
        text = f"What about {missing_aspect} is important for {query}?"

    if text.lower() in asked_norm:
        simple = f"What is {missing_aspect} for {query}?"
        if simple.lower() not in asked_norm:
            return simple
        return None

    return text

In [16]:
# generate answer
def generate_answer(question, docs, decomposition_path=None):
    import re

    reasoning_text = ""

    if decomposition_path:
        reasoning_text = "\n".join(decomposition_path)

    if not docs:
        return {
            "raw": "Insufficient information",
            "final": "Insufficient information"
        }

    # SAME evidence formatting style
    top_final = "\n".join(docs[:10])

    # EXACT system prompt style
    system_prompt_1 = (
        "Follow the given examples and Given the question and context, "
        "reasoning path, think step by step extract key segments from given evidence "
        "relevant to question and give rationale, by forming your own reasoning path "
        "preceded by [Answer]: and output final answer for the question using "
        "information from given evidences and give concise precise answer preceded "
        "by [Final Answer]:\n"
    )

    # SAME user prompt style
    user_prompt = """
    [Question]: When does monsoon season end in the state the area code 575 is located?
    [Answer]: The area code 575 is located in New Mexico. Monsoon season in New Mexico typically ends in mid-September.
    [Final Answer]: mid-September.

    [Question]: Who lived longer, Theodor Haecker or Harry Vaughan Watkins?
    [Answer]: Theodor Haecker was 65 years old when he died. Harry Vaughan Watkins was 69 years old when he died. Hence Harry Vaughan Watkins lived longer.
    [Final Answer]: Harry Vaughan Watkins.

    [Question]: What is the current official currency in the country where Ineabelle Diaz is a citizen?
    [Answer]: Ineabelle Diaz is from Puerto Rico, which is in the United States of America. The current official currency in the United States is the United States dollar.
    [Final Answer]: United States dollar.

    [Question]: Where was the person who founded the American Institute of Public Opinion in 1935 born?
    [Answer]: The person who founded the American Institute of Public Opinion in 1935 is George Gallup. George Gallup was born in Jefferson, Iowa.
    [Final Answer]: Jefferson.

    [Question]: What language is used by the director of Tiffany Memorandum?
    [Answer]: The director of Tiffany Memorandum is Sergio Grieco. Sergio Grieco speaks Italian.
    [Final Answer]: Italian.

    [Question]: What is the sports team the person played for who scored the first touchdown in Superbowl 1?
    [Answer]: The player that scored the first touchdown in Superbowl 1 is Max McGee. Max McGee played for the Green Bay Packers.
    [Final Answer]: Green Bay Packers.

    [Question]: The birth country of Jayantha Ketagoda left the British Empire when?
    [Answer]: The birth country of Jayantha Ketagoda is Sri Lanka. Sri Lanka left the British Empire on February 4, 1948.
    [Final Answer]: February 4, 1948.

    """ + (
            "Follow the above example and Given existing Reasoning path: "
            + reasoning_text
            + "\n the evidence, Evidence: "
            + top_final
            + "\n and use the most relevant information for the question "
            + "from the most relevant evidence from given Evidence: "
            + "and form your own correct reasoning path to derive the answer "
            + "thinking step by step preceded by [Answer]: "
            + "and subsequently give final answer as shown in above examples "
            + "preceded by [Final Answer]: for the Question: "
            + question
        )

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[
            {
                'role': 'system',
                'content': system_prompt_1
            },
            {
                'role': 'user',
                'content': user_prompt
            }
        ],
        options={
            'num_predict': 1000,
            'temperature': 0.2
        }
    )

    output = response['message'].get('content', '').strip()

    # safer extraction
    matches = re.findall(
        r"\[Final Answer\]:\s*(.+)",
        output,
        re.IGNORECASE
    )

    final_answer = matches[-1].strip() if matches else output.strip()

    return {
        "raw": output,
        "final": final_answer
    }

In [17]:
def run_iterative_pipeline_bm25(query, original_docs, max_steps=3):

    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    history = []
    all_docs = []
    base_aspects = extract_aspects(query)
    asked_followups = []
    resolved_aspects = []
    current_query = query

    for step in range(1, max_steps + 1):
        # Rerank FIRST using current query
        docs = rerank_docs(current_query, original_docs, top_k=5)

        # Collect docs for pooling
        all_docs.extend(docs)

        # Then generate answer
        history_text = []

        for h in history:
            if h.get("query"):
                history_text.append(f"Question: {h['query']}")
            if h.get("answer"):
                history_text.append(f"Intermediate Answer: {h['answer']}")

        answer_obj = generate_answer(
            current_query,
            docs,
            decomposition_path=history_text
        )

        answer = answer_obj["final"]
        raw_answer = answer_obj["raw"]

        combined_answer_parts = [
            item["answer"]
            for item in history
            if item.get("answer") and item["answer"] != "Insufficient information"
        ]
        if answer and answer != "Insufficient information":
            combined_answer_parts.append(answer)

        combined_answer = " ".join(combined_answer_parts).strip() or answer

        remaining_aspects = [a for a in base_aspects if a not in resolved_aspects]

        missing = find_missing_aspect(
            query,
            combined_answer,
            remaining_aspects,
            allow_intent=(step == 1)
        )

        step_record = {
            "step": step,
            "query": current_query,
            "answer": answer,
            "missing": missing,
            "followup": None
        }

        if missing not in ["NONE", "INTENT", "DETAIL"] and missing in remaining_aspects:
            resolved_aspects.append(missing)

        if missing == "NONE":
            history.append(step_record)
            break

        if current_query != query and answer == "Insufficient information":
            history.append(step_record)
            break

        if step >= max_steps:
            history.append(step_record)
            break

        followup = generate_aspect_followup(
            query,
            missing,
            remaining_aspects,
            asked_followups=asked_followups
        )

        if followup and "what specific information" in followup.lower():
            followup = f"What is the intended meaning of {query}?"

        if not followup:
            history.append(step_record)
            break

        if history:
            prev_followup = history[-1].get("followup")
            prev_answer = normalize(history[-1].get("answer", ""))
            curr_answer = normalize(answer)

            combined_prev = normalize(" ".join([
                item["answer"] for item in history
                if item.get("answer") and item["answer"] != "Insufficient information"
            ]))

            no_new_information = curr_answer and curr_answer in combined_prev
            repeated_followup = followup == prev_followup
            highly_similar_answer = (
                curr_answer[:140] == prev_answer[:140]
                if curr_answer and prev_answer else False
            )

            if repeated_followup or highly_similar_answer or no_new_information:
                history.append(step_record)
                break

        step_record["followup"] = followup
        history.append(step_record)

        asked_followups.append(followup)
        current_query = followup

    # Preserve order while removing duplicates
    seen = set()
    pooled_docs = []
    for d in all_docs:
        if d not in seen:
            seen.add(d)
            pooled_docs.append(d)

    # Re-rank pooled docs with original query
    reranked_pooled_docs = rerank_docs(query, pooled_docs)

    return history, docs, pooled_docs, reranked_pooled_docs

In [ ]:
test_sample = data[:5]

for item in test_sample:
    qid = item.get("_id", item.get("id"))
    query = item["question"]
    answer = item["answer"]

    raw_docs = bm25_data[qid]

    history, final_docs, pooled_docs, reranked_pooled_docs = run_iterative_pipeline_bm25(query, raw_docs)

    print("\nQUERY:", query)

    for step in history:
        print(step)

    print("Iterative (last step):", recall_at_k_2wiki(item, final_docs))
    print("Iterative (pooled):", recall_at_k_2wiki(item, pooled_docs))
    print("Iterative (pooled + rerank):", recall_at_k_2wiki(item, reranked_pooled_docs))


QUERY: Who is the mother of the director of film Polish-Russian War (Film)?
{'step': 1, 'query': 'Who is the mother of the director of film Polish-Russian War (Film)?', 'answer': 'Insufficient information', 'missing': 'INTENT', 'followup': 'What is the intended meaning of Who is the mother of the director of film Polish-Russian War (Film)??'}
{'step': 2, 'query': 'What is the intended meaning of Who is the mother of the director of film Polish-Russian War (Film)??', 'answer': 'Insufficient information', 'missing': 'identity', 'followup': None}
Iterative (last step): 0.5
Iterative (pooled): 0.5
Iterative (pooled + rerank): 0.5

QUERY: Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
{'step': 1, 'query': 'Which film came out first, Blind Shaft or The Mask Of Fu Manchu?', 'answer': 'Insufficient information', 'missing': 'INTENT', 'followup': 'What is the intended meaning of Which film came out first, Blind Shaft or The Mask Of Fu Manchu??'}
{'step': 2, 'query': 'What is t

# Recall

In [ ]:
# Not needed
import pandas as pd
from tqdm import tqdm

results = []

for item in tqdm(data, desc="Running full pipeline"):
    qid = item.get("_id", item.get("id"))
    query = item["question"]

    raw_docs = bm25_data[qid]

    # BM25
    bm25_score = recall_at_k_2wiki(item, raw_docs)

    # Baseline
    bq = generate_baseline_followup(query)
    b_docs = rerank_docs(bq, raw_docs)
    b_score = recall_at_k_2wiki(item, b_docs)

    # Decomposition
    dq = generate_decomposition_followup(query)
    d_docs = rerank_docs(dq, raw_docs)
    d_score = recall_at_k_2wiki(item, d_docs)

    # Yes/No
    yq = generate_yesno_followup(query)
    y_docs = rerank_docs(yq, raw_docs)
    y_score = recall_at_k_2wiki(item, y_docs)

    # Iterative
    history, final_docs, pooled_docs, reranked_pooled_docs = run_iterative_pipeline_bm25(query, raw_docs)
    
    iter_score = recall_at_k_2wiki(item, final_docs)
    iter_pool_score = recall_at_k_2wiki(item, pooled_docs, k=len(pooled_docs))
    iter_pool_rerank_score = recall_at_k_2wiki(item, reranked_pooled_docs)

    results.append({
        "qid": qid,
        "bm25": bm25_score,
        "baseline": b_score,
        "decomposition": d_score,
        "yesno": y_score,
        "iterative": iter_score,
        "iterative_pool": iter_pool_score,
        "iterative_pool_rerank": iter_pool_rerank_score,
        "steps": len(history)
    })

df = pd.DataFrame(results)

print("\nAVERAGE RESULTS:")
print(df[["bm25", "baseline", "decomposition", "yesno", "iterative", "iterative_pool","iterative_pool_rerank"]].mean())

df.to_csv("/kaggle/working/results_full.csv", index=False)

In [37]:
import os
print(os.listdir("/kaggle/working"))

['results_full.csv', '.virtual_documents', 'bm25_2wiki.json']


In [39]:
new_pool_scores = []

for item in tqdm(data, desc="Recomputing pooled recall"):
    qid = item.get("_id", item.get("id"))
    raw_docs = bm25_data[qid]

    _, _, pooled_docs, _ = run_iterative_pipeline_bm25(item["question"], raw_docs)

    score = recall_at_k_2wiki(item, pooled_docs, k=len(pooled_docs))
    new_pool_scores.append(score)

Recomputing pooled recall: 100%|██████████| 500/500 [30:27<00:00,  3.66s/it]


In [42]:
df["iterative_pool"] = new_pool_scores

In [43]:
summary = df[[
    "bm25",
    "baseline",
    "decomposition",
    "yesno",
    "iterative",
    "iterative_pool",
    "iterative_pool_rerank"
]].mean().round(4)

print("\nFINAL RESULTS:")
print(summary)


FINAL RESULTS:
bm25                     0.6100
baseline                 0.3900
decomposition            0.4610
yesno                    0.4400
iterative                0.5755
iterative_pool           0.6460
iterative_pool_rerank    0.6155
dtype: float64


In [44]:
df.to_csv("/kaggle/working/results_final.csv", index=False)

In [18]:
from collections import Counter

def clean_pred(text):
    import re

    text = text.strip()

    # extract final answer
    matches = re.findall(
        r"\[?Final Answer\]?:\s*(.*)",
        text,
        re.IGNORECASE
    )

    if matches:
        ans = matches[-1].strip()

        # empty final answer -> fallback to answer
        if ans == "":
            answer_matches = re.findall(
                r"\[?Answer\]?:\s*(.*)",
                text,
                re.IGNORECASE
            )

            if answer_matches:
                ans = answer_matches[-1].strip()

        ans = re.sub(
            r"^\[?Answer\]?:\s*",
            "",
            ans,
            flags=re.IGNORECASE
        )

        return ans.rstrip(" .")

    # fallback
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    return lines[-1].rstrip(" .") if lines else text


def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()


def exact_match(pred, gold):
    pred = normalize(pred)
    gold = normalize(gold)

    return int(pred == gold or pred in gold or gold in pred)


def f1_score(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


def evaluate(results):
    metrics = {
        "bm25": {"em": 0, "f1": 0},
        "baseline": {"em": 0, "f1": 0},
        "decomposition": {"em": 0, "f1": 0},
        "yesno": {"em": 0, "f1": 0},
        "iterative": {"em": 0, "f1": 0},
    }

    n = len(results)

    for item in results:
        gold = item["gold"]

        for method in metrics:
            pred = clean_pred(item[f"{method}_pred"])

            metrics[method]["em"] += exact_match(pred, gold)
            metrics[method]["f1"] += f1_score(pred, gold)

    for method in metrics:
        metrics[method]["em"] /= n
        metrics[method]["f1"] /= n

    return metrics

# Test accuracy on 20 samples

In [18]:
# not needed
from tqdm import tqdm

results = []

# 20 samples for testing
for i, item in enumerate(tqdm(data[:20], desc="Testing pipeline")):

    qid = item.get("_id", item.get("id"))
    query = item["question"]
    gold = item["answer"]

    if qid not in bm25_data:
        continue

    raw_docs = bm25_data[qid]

    # BM25
    bm25_docs = raw_docs[:5]
    bm25_pred = generate_answer(query, bm25_docs)

    # Baseline
    try:
        b_docs = rerank_docs(generate_baseline_followup(query), raw_docs)[:5]
    except:
        b_docs = raw_docs[:5]
    b_pred = generate_answer(query, b_docs)

    # Decomposition
    try:
        d_docs = rerank_docs(generate_decomposition_followup(query), raw_docs)[:5]
    except:
        d_docs = raw_docs[:5]
    d_pred = generate_answer(query, d_docs)

    # Yes/No
    try:
        y_docs = rerank_docs(generate_yesno_followup(query), raw_docs)[:5]
    except:
        y_docs = raw_docs[:5]
    y_pred = generate_answer(query, y_docs)

    # Iterative
    try:
        history, final_docs, pooled_docs, reranked_pooled_docs = run_iterative_pipeline_bm25(query, raw_docs)
        # iter_docs = pooled_docs
        iter_docs = reranked_pooled_docs[:5]
    except:
        iter_docs = raw_docs[:5]

    iter_pred = generate_answer(query, iter_docs)

    # STORE
    results.append({
        "gold": gold,
        "question": query,

        "bm25_pred": bm25_pred["final"],
        "bm25_raw": bm25_pred["raw"],

        "baseline_pred": b_pred["final"],
        "baseline_raw": b_pred["raw"],

        "decomposition_pred": d_pred["final"],
        "decomposition_raw": d_pred["raw"],

        "yesno_pred": y_pred["final"],
        "yesno_raw": y_pred["raw"],

        "iterative_pred": iter_pred["final"],
        "iterative_raw": iter_pred["raw"]
    })

print("\nDONE TEST RUN")

Testing pipeline: 100%|██████████| 20/20 [19:29<00:00, 58.45s/it]


DONE TEST RUN


In [19]:
with open("sample_results.json", "w") as f:
    json.dump(results, f, indent=2)

In [20]:
metrics = evaluate(results)

for method, scores in metrics.items():
    print(f"{method:<14} EM: {scores['em']:.3f}   F1: {scores['f1']:.4f}")

bm25           EM: 0.200   F1: 0.2380
baseline       EM: 0.200   F1: 0.2331
decomposition  EM: 0.250   F1: 0.2976
yesno          EM: 0.300   F1: 0.2946
iterative      EM: 0.150   F1: 0.1976


In [22]:
methods = ["bm25", "baseline", "decomposition", "yesno", "iterative"]

for r in results[:20]:

    print("\nQUESTION:", r["question"])

    print("GOLD:", repr(r["gold"]))

    for method in methods:
        print(f"\nMETHOD: {method}")
        print("PRED:", repr(clean_pred(r[f"{method}_pred"])))
        print("RAW:", r[f"{method}_raw"])

    print("\n" + "="*80)


QUESTION: Who is the mother of the director of film Polish-Russian War (Film)?
GOLD: 'Małgorzata Braunek'

METHOD: bm25
PRED: 'Unfortunately, we cannot determine who is the mother of the director of film "Polish-Russian War" based on the given evidence'
RAW: [Question]: Who is the mother of the director of film Polish-Russian War (Film)?

[Evidence]: 
Polish- Russian War( Wojna polsko- ruska) is a 2009 Polish film directed by Xawery Żuławski based on the novel Polish- Russian War under the white- red flag by Dorota Masłowska.

[Answer]: The director of the film "Polish-Russian War" is Xawery Żuławski. Since his mother's nationality is not mentioned in the given evidence, we cannot determine her identity.

However, another piece of information from the same evidence can be used to answer this question: 
Yanina Boleslavovna Zhejmo (29 May 1909 – 29 December 1987) was a Soviet actress of Polish origin. Her father was Polish and her mother was Russian.

[Answer]: Although Xawery Żuławski 

# Accuracy - All samples

In [45]:
# not needed
import json
from tqdm import tqdm

results = []

for i, item in enumerate(tqdm(data, desc="Running full pipeline")):

    qid = item.get("_id", item.get("id"))
    query = item["question"]
    gold = item["answer"]

    if qid not in bm25_data:
        continue

    raw_docs = bm25_data[qid]

    # BM25
    bm25_docs = raw_docs[:5]
    bm25_pred = generate_answer(query, bm25_docs)

    # Baseline
    try:
        b_docs = rerank_docs(generate_baseline_followup(query), raw_docs)[:5]
    except:
        b_docs = raw_docs[:5]
    b_pred = generate_answer(query, b_docs)

    # Decomposition
    try:
        d_docs = rerank_docs(generate_decomposition_followup(query), raw_docs)[:5]
    except:
        d_docs = raw_docs[:5]
    d_pred = generate_answer(query, d_docs)

    # Yes/No
    try:
        y_docs = rerank_docs(generate_yesno_followup(query), raw_docs)[:5]
    except:
        y_docs = raw_docs[:5]
    y_pred = generate_answer(query, y_docs)

    # Iterative
    try:
        history, final_docs, pooled_docs, reranked_pooled_docs = run_iterative_pipeline_bm25(query, raw_docs)
        # iter_docs = pooled_docs
        iter_docs = reranked_pooled_docs[:5]
    except:
        iter_docs = raw_docs[:5]

    iter_pred = generate_answer(query, iter_docs)

    # STORE
    results.append({
        "gold": gold,
        "question": query,

        "bm25_pred": bm25_pred["final"],
        "bm25_raw": bm25_pred["raw"],

        "baseline_pred": b_pred["final"],
        "baseline_raw": b_pred["raw"],

        "decomposition_pred": d_pred["final"],
        "decomposition_raw": d_pred["raw"],

        "yesno_pred": y_pred["final"],
        "yesno_raw": y_pred["raw"],

        "iterative_pred": iter_pred["final"],
        "iterative_raw": iter_pred["raw"]
    })

    # save every 5
    if (i + 1) % 5 == 0:
        with open("results_2wiki_accuracy_checkpoint.json", "w") as f:
            json.dump(results, f)

# final save
with open("results_2wiki_accuracy.json", "w") as f:
    json.dump(results, f, indent=2)


metrics = evaluate(results)

print("\nFINAL METRICS:")
for method, scores in metrics.items():
    print(f"{method:<14} EM: {scores['em']:.3f}   F1: {scores['f1']:.4f}")

Running full pipeline: 100%|██████████| 500/500 [5:25:30<00:00, 39.06s/it]  


FINAL METRICS:
bm25           EM: 0.252   F1: 0.2504
baseline       EM: 0.196   F1: 0.2005
decomposition  EM: 0.208   F1: 0.1964
yesno          EM: 0.178   F1: 0.1785
iterative      EM: 0.242   F1: 0.2462


# QA for iterative_pool and iterative_pool_rerank

In [22]:
# Additional evaluation for iterative pooling and reranking methods

def evaluate_extra(results):

    metrics = {
        "iter_pool": {"em": 0, "f1": 0},
        "iter_pool_rerank": {"em": 0, "f1": 0},
    }

    n = len(results)

    for item in results:
        gold = item["gold"]

        for method in metrics:
            pred = clean_pred(item[f"{method}_pred"])

            metrics[method]["em"] += exact_match(pred, gold)
            metrics[method]["f1"] += f1_score(pred, gold)

    for method in metrics:
        metrics[method]["em"] /= n
        metrics[method]["f1"] /= n

    return metrics

In [27]:
# Evaluate the iterative pooling and reranking methods

import re
import json
from tqdm import tqdm

results_extra = []


for i, item in enumerate(tqdm(data, desc="Testing iterative pool methods")):

    qid = item.get("_id", item.get("id"))
    query = item["question"]
    gold = item["answer"]

    if qid not in bm25_data:
        continue

    raw_docs = bm25_data[qid]

    try:
        history, final_docs, pooled_docs, reranked_pooled_docs = run_iterative_pipeline_bm25(
            query,
            raw_docs
        )

        iter_pool_pred = generate_answer(
            query,
            pooled_docs[:5]
        )

        iter_pool_rerank_pred = generate_answer(
            query,
            reranked_pooled_docs[:5]
        )

    except Exception as e:

        print("\nERROR:")
        print(e)

        iter_pool_pred = {
            "final": "Unknown",
            "raw": str(e)
        }

        iter_pool_rerank_pred = {
            "final": "Unknown",
            "raw": str(e)
        }

    results_extra.append({
        "gold": gold,
        "question": query,

        "iter_pool_pred": iter_pool_pred["final"],
        "iter_pool_raw": iter_pool_pred["raw"],

        "iter_pool_rerank_pred": iter_pool_rerank_pred["final"],
        "iter_pool_rerank_raw": iter_pool_rerank_pred["raw"]
    })

# save test results

with open("results_iterpool_test.json", "w", encoding="utf-8") as f:
    json.dump(results_extra, f, ensure_ascii=False, indent=2)

# EVALUATION

extra_metrics = evaluate_extra(results_extra)

print("\nTEST METRICS:\n")

for method, vals in extra_metrics.items():
    print(
        f"{method:<20} EM: {vals['em']:.3f}   F1: {vals['f1']:.4f}"
    )

Testing iterative pool methods: 100%|██████████| 500/500 [3:33:14<00:00, 25.59s/it]  


TEST METRICS:

iter_pool            EM: 0.244   F1: 0.2406
iter_pool_rerank     EM: 0.260   F1: 0.2555
